# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies, optionally installs/runs Ollama, and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and optionally start Ollama. Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1: Environment setup + .env generator + optional Ollama installer
# Run this cell first. It installs packages and provides functions to write .env and (optionally) install/run Ollama.

# Install required packages
!pip install -q pypdf langchain langchain-community langchain-openai python-dotenv chromadb ipywidgets sentence-transformers

# Enable ipywidgets in Colab (may need refresh)
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Imports
import os
import json
import subprocess
import time
from pathlib import Path
from dotenv import load_dotenv, set_key
from uuid import uuid4

# .env helper
ENV_PATH = Path('.env')

def write_env(use_backend="openai", openai_api_key="", hf_embedding_model="all-MiniLM-L6-v2"):
    """
    Writes .env toggles:
    - BACKEND=ollama|openai
    - OPENAI_API_KEY (if using openai)
    - HF_EMBEDDING_MODEL (for local embeddings)
    """
    lines = [
        f"BACKEND={use_backend}",
        f"HF_EMBEDDING_MODEL={hf_embedding_model}",
        f"OPENAI_API_KEY={openai_api_key or ''}"
    ]
    ENV_PATH.write_text("
".join(lines))
    load_dotenv(dotenv_path=ENV_PATH)
    return str(ENV_PATH)

# Basic Ollama installer/runner (best-effort - may require manual tweaks in Colab)
def install_and_run_ollama(pull_model="gemma4:4b"):
    """
    Attempts to install ollama and run 'ollama serve' in background,
    then pulls the required model. This may require elevated permissions
    or manual steps depending on Colab environment.
    """
    try:
        print("Attempting to install Ollama (best-effort). This may fail on some Colab instances.")
        subprocess.run("curl -fsSL https://ollama.com/install | sh", shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print("Ollama install script failed:", e)
        print("You may need to install Ollama manually or run Colab on a compatible environment.")
        return False

    try:
        # Start ollama serve in background
        print("Starting `ollama serve` in background...")
        subprocess.Popen(f"nohup ollama serve &> ollama_serve.log &", shell=True)
        time.sleep(2)
        # Pull model
        print(f"Pulling model {pull_model}...")
        subprocess.run(f"ollama pull {pull_model}", shell=True, check=True)
        print("Ollama should be running. Check 'ollama_serve.log' for logs.")
        return True
    except Exception as e:
        print("Failed to start or pull Ollama model:", e)
        return False

# Quick .env example write (default to openai; user can toggle in UI)
write_env(use_backend="openai", openai_api_key="", hf_embedding_model="all-MiniLM-L6-v2")
print("Wrote .env with default settings (BACKEND=openai). Update via UI in the next cell.")

In [ ]:
# Colab Cell 2: RAG app with ipywidgets UI
# Paste and run after Cell 1. The UI allows switching backend, uploading a PDF, asking questions, and generating summaries.

# Imports for core logic
import os
import io
import threading
from pypdf import PdfReader
from dotenv import load_dotenv
from IPython.display import display, Markdown
import ipywidgets as widgets
from uuid import uuid4

# LangChain & vectorstore imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI as LangOpenAI
from langchain_community.llms import Ollama

# Load environment if present
load_dotenv(".env")

# Utility: read PDF bytes -> text
def pdf_bytes_to_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    text_pages = []
    for p in reader.pages:
        try:
            txt = p.extract_text() or ""
        except Exception:
            txt = ""
        text_pages.append(txt)
    return "

".join(text_pages)

# Build vectorstore from PDF bytes, returns Chroma retriever
def build_vectorstore_from_pdf(pdf_bytes, persist_dir=None):
    # load .env toggles
    load_dotenv(".env")
    backend = os.getenv("BACKEND", "openai").lower()
    hf_model = os.getenv("HF_EMBEDDING_MODEL", "all-MiniLM-L6-v2")
    openai_key = os.getenv("OPENAI_API_KEY", "
,
,
,
,
,
,
,
,
ollama":
        # local huggingface embeddings
        embedding_fn = HuggingFaceEmbeddings(model_name=hf_model)
    else:
        # OpenAI embeddings
        embedding_fn = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_key)

    # Persist directory or temp
    persist_dir = persist_dir or f"chroma_store_{uuid4().hex}"
    vectordb = Chroma.from_texts(documents=docs, embedding=embedding_fn, persist_directory=persist_dir)
    return vectordb.as_retriever(search_kwargs={"k": 4}), persist_dir

# LLM selection based on backend
def get_llm():
    load_dotenv(".env")
    backend = os.getenv("BACKEND", "openai").lower()
    openai_key = os.getenv("OPENAI_API_KEY", "
,
ollama":
        # use Ollama LLM (assumes ollama serve is running)
        return Ollama(model="gemma4:4b", temperature=0.0)
    else:
        # OpenAI LLM via LangChain wrapper
        return LangOpenAI(model_name="gpt-4o-mini", openai_api_key=openai_key, temperature=0.0)

# Domain-aware prompt template enforcing strict grounding
BASE_PROMPT = """
You are a helpful assistant that strictly answers only from the provided CONTEXT. Use the CONTEXT snippets to answer the QUESTION.
Domain: {domain}

RULES:
- Use only the provided CONTEXT to answer.
- If the answer cannot be found verbatim or inferred from the CONTEXT, respond exactly: "I cannot find that information in the provided document."
- Keep answers concise and focused.
- When the domain is "Law", emphasize definitions, clauses, and liabilities.
- When the domain is "Telecom" or "Media", emphasize technical specs, service terms, or metrics.
- When the domain is "General", provide concise, neutral answers.

CONTEXT:
{context}

QUESTION: {question}

Answer:
"""

PROMPT_TEMPLATE = PromptTemplate(template=BASE_PROMPT, input_variables=["context", "question", "domain"])

# Query function
def query_with_rag(retriever, question, domain):
    llm = get_llm()
    # Build RetrievalQA using a simple 'stuff' chain and our PromptTemplate
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, chain_type_kwargs={"prompt": PROMPT_TEMPLATE})
    result = qa_chain.run({"query": question, "domain": domain})
    # The chain returns text; ensure the exact fallback phrase is used if model deviated
    if "I cannot find that information in the provided document." in result:
        return "I cannot find that information in the provided document."
    return result

# Summary function: retrieve top docs and ask LLM to synthesize structured summary
SUMMARY_PROMPT = """
You are a summarization assistant. Use only the CONTEXT below to produce a structured summary.
Domain: {domain}

If important details are missing in CONTEXT, state "I cannot find that information in the provided document."
Context:
{context}

Produce a concise structured summary with headings where useful.
"""
SUMMARY_TEMPLATE = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["context", "domain"])

def generate_summary(retriever, domain):
    llm = get_llm()
    docs = retriever.get_relevant_documents("summary")
    combined = "

".join([d.page_content for d in docs])
    prompt = SUMMARY_TEMPLATE.format(context=combined, domain=domain)
    llm_response = llm(prompt)
    if "I cannot find that information in the provided document." in llm_response:
        return "I cannot find that information in the provided document."
    return llm_response

# -----------------------------
# UI using ipywidgets
# -----------------------------
# Widgets
backend_dropdown = widgets.Dropdown(options=[("OpenAI (Cloud)", "openai"), ("Ollama (Local)", "ollama")],
                                    value=os.getenv("BACKEND", "openai"),
                                    description="Backend:")

openai_key_text = widgets.Password(value=os.getenv("OPENAI_API_KEY", ""), description="OpenAI Key:")
hf_model_text = widgets.Text(value=os.getenv("HF_EMBEDDING_MODEL", "all-MiniLM-L6-v2"),
                             description="HF Embedding:")

domain_dropdown = widgets.Dropdown(options=["Media", "Law", "Telecom", "General"], value="General", description="Domain:")
file_uploader = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")

ask_text = widgets.Text(value="", description="Ask a Question:", layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask", button_style="primary")
summary_button = widgets.Button(description="Generate Summary", button_style="info")
install_ollama_button = widgets.Button(description="Install & Run Ollama", button_style="warning")
status_out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="10px"))
console_out = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="10px"))

# State holders
_state = {"retriever": None, "persist_dir": None, "pdf_filename": None}

# Handlers
def on_backend_change(change):
    if change["new"] == "openai":
        openai_key_text.layout.display = None
    else:
        openai_key_text.layout.display = "none"

backend_dropdown.observe(on_backend_change, names="value")
# initialize display state
on_backend_change({"new": backend_dropdown.value})

def on_install_ollama_clicked(b):
    with status_out:
        status_out.clear_output()
        print("Attempting Ollama install and run (this may take a while)...")
    def worker():
        ok = install_and_run_ollama(pull_model="gemma4:4b")
        with status_out:
            if ok:
                print("Ollama installation/launch attempted. Check 'ollama_serve.log' for details.")
            else:
                print("Ollama install/run failed or requires manual steps.")
    threading.Thread(target=worker).start()

install_ollama_button.on_click(on_install_ollama_clicked)

def on_write_env():
    backend = backend_dropdown.value
    write_env(use_backend=backend, openai_api_key=openai_key_text.value, hf_embedding_model=hf_model_text.value)
    with status_out:
        status_out.clear_output()
        print(f"Wrote .env with BACKEND={backend} and HF_EMBEDDING_MODEL={hf_model_text.value}")

def on_upload_and_index():
    if len(file_uploader.value) == 0:
        with status_out:
            status_out.clear_output()
            print("Please upload a PDF first.")
        return
    # Get uploaded file
    uploaded = next(iter(file_uploader.value.values()))
    fname = uploaded["metadata"]["name"]
    b = uploaded["content"]
    with status_out:
        status_out.clear_output()
        print(f"Indexing {fname} ... This may take a moment.")
    def worker():
        try:
            on_write_env()
            retriever, persist_dir = build_vectorstore_from_pdf(b)
            _state["retriever"] = retriever
            _state["persist_dir"] = persist_dir
            _state["pdf_filename"] = fname
            with status_out:
                status_out.clear_output()
                print(f"Indexed '{fname}' into Chroma at {persist_dir}. Ready to query.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Indexing failed:", e)
    threading.Thread(target=worker).start()

# Automatically index when file uploaded
def on_file_upload_change(change):
    if change['new']:
        on_upload_and_index()

file_uploader.observe(on_file_upload_change, names='value')

def display_answer(text):
    console_out.clear_output()
    with console_out:
        display(Markdown(text))

def on_ask_clicked(b):
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No document indexed. Upload a PDF first.")
        return
    question = ask_text.value.strip()
    if not question:
        with status_out:
            status_out.clear_output()
            print("Please type a question.")
        return

    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print("Processing question...")

    def worker():
        try:
            ans = query_with_rag(_state["retriever"], question, domain)
            display_answer(ans)
            with status_out:
                status_out.clear_output()
                print("Done.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Query failed:", e)
    threading.Thread(target=worker).start()

ask_button.on_click(on_ask_clicked)

def on_summary_clicked(b):
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No document indexed. Upload a PDF first.")
        return
    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print("Generating summary...")

    def worker():
        try:
            summ = generate_summary(_state["retriever"], domain)
            display_answer(summ)
            with status_out:
                status_out.clear_output()
                print("Summary generated.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Summary failed:", e)
    threading.Thread(target=worker).start()

summary_button.on_click(on_summary_clicked)

# Layout composition
config_box = widgets.VBox([
    widgets.HTML("<b>Configuration</b>"),
    backend_dropdown,
    openai_key_text,
    hf_model_text,
    widgets.HBox([install_ollama_button, widgets.Button(description="Write .env", on_click=lambda b: on_write_env())])
])

upload_box = widgets.VBox([
    widgets.HTML("<b>Upload Document</b>"),
    domain_dropdown,
    file_uploader
])

action_box = widgets.VBox([
    widgets.HTML("<b>Actions</b>"),
    widgets.HBox([ask_text, ask_button]),
    widgets.HTML("<i>Or</i>"),
    summary_button
])

ui = widgets.VBox([
    widgets.HBox([config_box, upload_box, action_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Status</b>"),
    status_out,
    widgets.HTML("<b>Console Output</b>"),
    console_out
])

display(ui)
print("UI ready. Use the Configuration panel to set backend/key, upload a PDF, then Ask or Generate Summary.")